In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
"""Rank days by lightning activity over Italy using one LI product per hour (cheap screening)."""
import logging
from datetime import datetime
from itertools import islice
from pathlib import Path

import numpy as np
import pandas as pd
from satpy import Scene
from tqdm import tqdm

from src.download_data import download_entry, get_datastore, search, select_entries
from src.preprocessing import AREA_EXTENT

LI_COLLECTION_ID = "EO:EUM:DAT:0686"
SCREENING_PERIODS = [  # edit: candidate periods, away from 18-22 July
    (datetime(2026, 7, 1), datetime(2026, 7, 17)),
    (datetime(2026, 7, 23), datetime(2026, 8, 1)),
]
HOURLY_STEP = 6  # one 10-minute LI product per hour
OUTPUT_DIR = Path("../data/raw_li_screening")
log = logging.getLogger(__name__)


def italy_accumulation(li_path: Path):
    """Start time and summed flash accumulation inside the (slightly generous) crop over Italy."""
    scene = Scene(filenames=[str(li_path)], reader="li_l2_nc")
    scene.load(["flash_accumulation"])
    italy_field = scene.crop(ll_bbox=AREA_EXTENT)["flash_accumulation"].values
    return scene.start_time, float(np.nansum(italy_field))


def main() -> None:
    logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    datastore = get_datastore()
    area = ",".join(str(value) for value in AREA_EXTENT)

    sampled_rows = []
    for period_start, period_end in SCREENING_PERIODS:
        products = search(datastore, LI_COLLECTION_ID, period_start, period_end, area)
        sampled_count = len(range(0, products.total_results, HOURLY_STEP))
        for product in tqdm(islice(products, 0, None, HOURLY_STEP), total=sampled_count, desc=str(period_start.date())):
            entry_names = select_entries(list(product.entries))
            li_path = download_entry(product, entry_names[0], OUTPUT_DIR)
            if li_path is None:
                continue
            sampled_time, accumulation = italy_accumulation(li_path)
            sampled_rows.append({"day": sampled_time.date().isoformat(), "accumulation": accumulation})

    daily = pd.DataFrame(sampled_rows).groupby("day")["accumulation"].agg(["sum", "count"])
    daily["estimated_total"] = daily["sum"] * HOURLY_STEP  # ranking only: 10 of every 60 minutes are sampled
    print(daily.sort_values("estimated_total", ascending=False).round(0))

In [3]:
main()

2026-10-23 14:06:59,374 INFO Token expires 2026-10-24 12:21:13.079918
2026-10-23 14:07:01,973 INFO Found 2294 products in EO:EUM:DAT:0686
2026-07-01: 100%|██████████| 383/383 [01:49<00:00,  3.50it/s]
2026-10-23 14:08:54,946 INFO Found 1294 products in EO:EUM:DAT:0686
2026-07-23: 100%|██████████| 216/216 [09:53<00:00,  2.75s/it]

                 sum  count  estimated_total
day                                         
2026-07-01  160126.0     24         960755.0
2026-07-02  155895.0     24         935369.0
2026-07-26  147979.0     23         887876.0
2026-07-11   88224.0     24         529345.0
2026-07-15   75262.0     24         451572.0
2026-07-27   61132.0     24         366793.0
2026-07-03   53220.0     24         319319.0
2026-07-24   33671.0     24         202023.0
2026-07-14   32077.0     24         192462.0
2026-07-10   30452.0     23         182710.0
2026-07-31   28397.0     24         170379.0
2026-07-23   28073.0     24         168436.0
2026-07-16   24960.0     24         149759.0
2026-07-25   22995.0     24         137970.0
2026-07-12   21281.0     24         127688.0
2026-07-05   10084.0     24          60506.0
2026-07-06    9687.0     23          58125.0
2026-07-08    8154.0     24          48921.0
2026-07-28    7193.0     24          43158.0
2026-07-13    6275.0     24          37652.0
2026-07-29